In [1]:
# ===========================================
# Modular Napari QC + Interactive Watershed
# ===========================================

import os
import numpy as np
from tifffile import imread
from skimage import filters, segmentation
import napari
from magicgui import magicgui

In [2]:
# -------------------------------
# 1. Utility: Load image in CHW
# -------------------------------
def load_image(path):
    """Load an image and convert to CHW float32 format for Napari."""
    img = imread(path)
    # RGB/RGBA → CHW
    if img.ndim == 3 and img.shape[-1] in [3,4]:
        img = np.transpose(img, (2, 0, 1))
    # Single-channel → add channel axis
    elif img.ndim == 2:
        img = img[np.newaxis, ...]
    return img.astype(np.float32)


In [9]:
# -------------------------------
# 2. Paths
# -------------------------------
image_folder   = r"T:\Orthopaedics\Lab Imaging Data\mmazur\NT_Validation_Study4\Rb41\Rb41_Left_E\40X_TIFF"
overlay_folder = r"T:\Orthopaedics\Lab Imaging Data\mmazur\NT_Validation_Study4\Rb41\Rb41_Left_E\40X_Segmented"

In [10]:
# -------------------------------
# 3. Load all images
# -------------------------------
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))])
overlay_files = sorted([f for f in os.listdir(overlay_folder) if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))])

images = [load_image(os.path.join(image_folder, f)) for f in image_files]
overlays = [load_image(os.path.join(overlay_folder, f)) for f in overlay_files]

for i, (img, overlay) in enumerate(zip(images, overlays)):
    print(f"Image {i}: shape={img.shape}, dtype={img.dtype}, min={img.min()}, max={img.max()}")
    print(f"Overlay {i}: shape={overlay.shape}, dtype={overlay.dtype}, min={overlay.min()}, max={overlay.max()}")
    print("-" * 50)

OME series: incompatible page shape (1024, 1440, 3); expected (3, 1024, 1440)
OME series: incompatible page shape (1024, 1440, 3); expected (3, 1024, 1440)
OME series: incompatible page shape (1024, 1440, 3); expected (3, 1024, 1440)


Image 0: shape=(3, 1024, 1440), dtype=float32, min=14.0, max=224.0
Overlay 0: shape=(3, 1024, 1280), dtype=float32, min=0.0, max=255.0
--------------------------------------------------
Image 1: shape=(3, 1024, 1440), dtype=float32, min=16.0, max=225.0
Overlay 1: shape=(3, 1024, 1280), dtype=float32, min=0.0, max=255.0
--------------------------------------------------
Image 2: shape=(3, 1024, 1440), dtype=float32, min=27.0, max=225.0
Overlay 2: shape=(3, 1024, 1280), dtype=float32, min=0.0, max=255.0
--------------------------------------------------


In [14]:
# -------------------------------
# 4. Napari viewer with translucent overlay outside mask
# -------------------------------

import napari
import numpy as np

# Create Napari viewer
viewer = napari.Viewer(title="Segmentation Check")

# Start with first image
if images and overlays:
    img_layer = viewer.add_image(
        images[0],
        name='Original',
        blending='additive',
        colormap=None
    )
    
    # Create a black translucent mask outside the segmentation
    # Assume overlay is a label image (0 = background, >0 = segmented)
    mask = overlays[0][0] > 0  # first channel if CHW
    translucent = np.ones_like(mask, dtype=np.float32)  # full brightness
    translucent[mask] = 0  # zero where mask exists (segmentation visible)
    
    overlay_layer = viewer.add_image(
        translucent,
        name='Darkened Outside Mask',
        opacity=0.5,            # adjust transparency
        blending='additive',
        colormap='black'
    )

# Navigation with Left/Right keys
def update_layers(event):
    if event.key == 'Right':
        update_layers.index = (update_layers.index + 1) % len(images)
    elif event.key == 'Left':
        update_layers.index = (update_layers.index - 1) % len(images)
    else:
        return
    
    img_layer.data = images[update_layers.index]
    
    mask = overlays[update_layers.index][0] > 0
    translucent = np.ones_like(mask, dtype=np.float32)
    translucent[mask] = 0
    overlay_layer.data = translucent

update_layers.index = 0
viewer.bind_key('Right', update_layers)
viewer.bind_key('Left', update_layers)

napari.run()

In [7]:
# -------------------------------
# Navigation slider
# -------------------------------
@magicgui(auto_call=True, slider={"max": len(images)-1})
def navigate(slider=0):
    i = int(slider)
    # Update image and overlay layers
    img_layer.data = images[i]
    overlay_layer.data = overlays[i]
    y_offset = (images[i].shape[1] - overlays[i].shape[1]) // 2
    x_offset = (images[i].shape[2] - overlays[i].shape[2]) // 2
    overlay_layer.translate = (y_offset, x_offset)

viewer.window.add_dock_widget(navigate, area='right')
napari.run()